In [ ]:
import requests


token = ""
deployment_url = "https://deployments.astro-stg.adp.tredence_analytics.com/marketing-data-analytics-master/airflow"
dag_id = "Lumos_TRG"
# response = requests.post(
#     url=f"{deployment_url}/api/v1/dags/{dag_id}/dagRuns",
#     headers={
#         "Authorization": f"Bearer {token}",
#         "Content-Type": "application/json"
#     },
#     data='{}'
# )
# print(response.json())
# Prints metadata of the dag run that was just triggered


In [ ]:
import json
import urllib.parse
import os
import requests
import socket
import time
from urllib.parse import urlparse
from typing import Dict, Any, Optional, Tuple

print('Loading function')

session = requests.Session()


def trigger_airflow_dag(url, headers, payload):

    print(":::::: Inside trigger_airflow_dag ::::::")
    
    try:

        response = session.post(
            url,
            headers=headers,
            json=payload,
            timeout=90,
            stream=False
        )
        
        print("Response: {}".format(response.status_code))
        return response
        
    except (requests.exceptions.ConnectTimeout, 
            requests.exceptions.ConnectionError) as e:
        raise ConnectionError("Request failed: {}".format(e))
           
    except requests.exceptions.RequestException as e:
        raise ConnectionError("Request failed: {}".format(e))

In [ ]:
# Format Airflow URL
airflow_url = "https://deployments.astro-dev.adp.tredence_analytics.com/marketing-data-analytics-master/airflow/api/v1/dags/{dag_id}/dagRuns".format(dag_id='actual_po_driver_edh')
parsed_url = urlparse(airflow_url)

print("Target: {} - {}".format('DEV', airflow_url))

# Prepare request
headers = {
    'Content-Type': 'application/json',
    'Accept': 'application/json',
    'Authorization': f"Bearer ea01c3cda3771eb81defcb7170be95ef",
    'User-Agent': 'AWS-Lambda/1.0'
}

print("Request headers: {}".format(headers))

payload = {
    "conf": {
        "s3_object_name": 's3://com.tredence_analytics.marketing.mars.prd.ue1.landing-zone/ariba_spend_edh/marketing_spend/dt=20250806/MARKETING_SPEND_2.csv',
        "partition": '20250806'
    }
}
print("Request payload: {}".format(payload))

# Trigger DAG
response = trigger_airflow_dag(airflow_url, headers, payload)

print(response.json())